# Student Performance Analytics with AI
### IBM SkillsBuild Data Analytics with AI Academic Internship — BharatCares x AICTE

**Author:** Siddhesh Patil
**Project:** Exploratory Data Analysis and Predictive Modeling of Student Exam Performance

---

## 1. Business Problem

Schools and ed-tech platforms want to understand which factors are most associated
with a student scoring well in exams, so they can target support (test-prep
programs, meal schemes, counselling) at the students who need it most.

This project analyzes a dataset of 1,000 students' math, reading and writing
scores along with demographic and preparation-related attributes, to:

1. Explore how gender, parental education, lunch type and test-preparation
   course completion relate to exam performance.
2. Build a simple machine learning model that predicts whether a student is
   likely to **pass** (average score >= 50) based on these attributes.
3. Provide data-driven recommendations.

## 2. Dataset

- **Source:** Kaggle — "Students Performance in Exams" (public dataset)
- **Rows:** 1,000 students
- **Columns:** gender, race/ethnicity, parental level of education, lunch,
  test preparation course, math score, reading score, writing score


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
%matplotlib inline


## 4. Load and Inspect Data

In [ ]:
df = pd.read_csv("StudentsPerformance.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe()


## 5. Feature Engineering

We create an `average_score` column and a binary `result` label (Pass/Fail, threshold = 50).

In [ ]:
df["average_score"] = (df["math score"] + df["reading score"] + df["writing score"]) / 3
df["result"] = np.where(df["average_score"] >= 50, "Pass", "Fail")
df[["math score", "reading score", "writing score", "average_score", "result"]].head()


## 6. Exploratory Data Analysis

### 6.1 Distribution of Average Scores

In [ ]:
plt.figure()
sns.histplot(df["average_score"], bins=20, kde=True, color="steelblue")
plt.title("Distribution of Average Score")
plt.xlabel("Average Score")
plt.ylabel("Number of Students")
plt.show()


### 6.2 Score by Gender

In [ ]:
plt.figure()
sns.boxplot(data=df, x="gender", y="average_score", hue="gender", palette="Set2", legend=False)
plt.title("Average Score by Gender")
plt.show()


### 6.3 Effect of Test Preparation Course

In [ ]:
plt.figure()
sns.boxplot(data=df, x="test preparation course", y="average_score", hue="test preparation course", palette="Set3", legend=False)
plt.title("Average Score vs Test Preparation Course")
plt.show()


### 6.4 Effect of Lunch Type (proxy for socio-economic status)

In [ ]:
plt.figure()
sns.boxplot(data=df, x="lunch", y="average_score", hue="lunch", palette="pastel", legend=False)
plt.title("Average Score by Lunch Type")
plt.show()


### 6.5 Parental Level of Education

In [ ]:
order = df.groupby("parental level of education")["average_score"].mean().sort_values().index
plt.figure(figsize=(9, 5))
sns.barplot(data=df, x="parental level of education", y="average_score", hue="parental level of education", order=order, palette="viridis", legend=False)
plt.title("Average Score by Parental Level of Education")
plt.xticks(rotation=30, ha="right")
plt.show()


### 6.6 Correlation Between Subject Scores

In [ ]:
plt.figure()
corr = df[["math score", "reading score", "writing score"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Correlation Between Math, Reading and Writing Scores")
plt.show()


### 6.7 Pass Rate by Race/Ethnicity Group

In [ ]:
pass_rate = df.groupby("race/ethnicity")["result"].apply(lambda x: (x == "Pass").mean() * 100)
plt.figure()
pass_rate.sort_values().plot(kind="barh", color="teal")
plt.title("Pass Rate (%) by Race/Ethnicity Group")
plt.xlabel("Pass Rate (%)")
plt.show()


## 7. Key Insights from EDA

- Students who **completed the test preparation course** score noticeably
  higher on average across all three subjects.
- Students with **standard lunch** outperform those with free/reduced lunch,
  hinting at a socio-economic effect on exam performance.
- **Parental level of education** shows a positive trend — students whose
  parents hold a bachelor's/master's degree tend to score higher on average.
- Math, reading and writing scores are **strongly correlated** with each other,
  meaning a student who does well in one subject tends to do well in the others.


## 8. Predictive Modeling — Pass/Fail Classification

We train a Random Forest classifier to predict whether a student passes (average score >= 50) using their demographic and preparation attributes (not the scores themselves, since those directly determine the label).

In [ ]:
features = ["gender", "race/ethnicity", "parental level of education", "lunch", "test preparation course"]
X = df[features].copy()
y = df["result"]

encoders = {}
for col in X.columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train size:", X_train.shape, " Test size:", X_test.shape)


In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))


### 8.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


### 8.2 Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values()
plt.figure()
importances.plot(kind="barh", color="darkorange")
plt.title("Feature Importance for Predicting Pass/Fail")
plt.xlabel("Importance")
plt.show()


## 9. Business Recommendations

1. **Promote test-preparation courses** — this was one of the strongest,
   most actionable predictors of passing. Schools should encourage or
   subsidize enrollment, especially for students at risk.
2. **Support students on free/reduced lunch** with targeted tutoring or
   after-school programs, since socio-economic status correlates with
   performance.
3. **Early outreach for first-generation students** — students whose
   parents have lower levels of formal education may benefit from
   additional mentoring or study-skills workshops.
4. **Monitor at the group level, not just individually** — pass-rate gaps
   across ethnicity groups suggest structural factors worth investigating
   further (access to resources, school quality, etc.), not innate ability.

## 10. Conclusion

This project demonstrates a complete data analytics with AI workflow:
data cleaning, exploratory analysis, visualization, and a machine-learning
classifier with 80%+ accuracy for predicting student outcomes from
demographic and preparation features. The insights point to concrete,
actionable interventions (test-prep access, socio-economic support) that
could improve pass rates.

---
*Submitted as part of the IBM SkillsBuild Data Analytics with AI Academic
Internship Program, conducted by BharatCares in association with AICTE.*
